# Set up Monday's tables (week 3, day 1)

For the trainer. Run this notebook once in the class workspace before Monday, from your Git folder of the course: `week-03/day-01/setup/setup.ipynb`. The numbered steps, and what to do before and after it, are in `README.md` beside it.

It makes the schema `case_selection` in the course catalog, and five tables in it: `returns` (40 rows), `third_party_reports` (40), `exam_outcomes` (24), `income_bands` (40) and `case_features` (40). The rows and the SQL are in `course_data.py` beside this notebook. The course made up every row: no name, number or amount belongs to a real taxpayer. Running it again is safe: it makes each table again, and puts back any comment a learner saved.

Each code cell has a number on its first line. Run cells 1 to 6 in order. Cell 7 is a choice: read its first lines.

In [ ]:
# 1. Settings. Set CATALOG to the course catalog, and CLASS_GROUP to the group that holds the learners.
CATALOG = "workspace"
CLASS_GROUP = "account users"

import os
try:
    _path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _folder = os.path.dirname(_path if _path.startswith("/Workspace") else "/Workspace" + _path)
except NameError:  # outside Databricks: the folder the notebook runs in
    _folder = os.getcwd()
course = {"__name__": "course_data"}
exec(open(os.path.join(_folder, "course_data.py"), encoding="utf-8").read(), course)
print("The tables go in", f"{CATALOG}.{course['SCHEMA']}", "and", CLASS_GROUP, "gets read access in cell 6.")

In [ ]:
# 2. The schema and the three base tables: returns, third_party_reports, exam_outcomes.
for statement in course["base_statements"](CATALOG):
    spark.sql(statement)
print("Made returns, third_party_reports and exam_outcomes, with their comments and tags.")

In [ ]:
# 3. income_bands, computed in Python. Lineage sees the read of returns and the write of
#    income_bands as two separate steps, so it cannot link income_band to total_income.
schema = f"{CATALOG}.{course['SCHEMA']}"
spark.sql(course["create_table"](CATALOG, "income_bands", [])[0])
bands = spark.table(f"{schema}.returns").select("return_id", "total_income").toPandas()
bands["income_band"] = bands["total_income"].map(course["income_band"])
spark.createDataFrame(bands[["return_id", "income_band"]]).write.mode("append").saveAsTable(f"{schema}.income_bands")
print("Made income_bands:", len(bands), "rows.")

In [ ]:
# 4. case_features, built by one SQL query on Databricks, so Unity Catalog records its lineage.
for statement in course["features_statements"](CATALOG):
    spark.sql(statement)
print("Made case_features: income_gap, interest_gap and income_band.")

In [ ]:
# 5. Read back what the learners will see. Expected: returns 40, third_party_reports 40,
#    exam_outcomes 24, income_bands 40, case_features 40; total_income, interest_reported and
#    payer_count have no comment; taxpayer_name and taxpayer_tin carry the tag data_class = pii.
schema = f"{CATALOG}.{course['SCHEMA']}"
for table in ["returns", "third_party_reports", "exam_outcomes", "income_bands", "case_features"]:
    count = spark.sql(f"SELECT COUNT(*) AS n FROM {schema}.{table}").first()["n"]
    empty = [r["column_name"] for r in spark.sql(
        f"SELECT column_name FROM {CATALOG}.information_schema.columns WHERE table_schema = '{course['SCHEMA']}' "
        f"AND table_name = '{table}' AND comment IS NULL").collect()]
    print(f"{table}: {count} rows; no comment on: {', '.join(empty) or 'none'}")
for r in spark.sql(f"SELECT table_name, column_name, tag_name, tag_value FROM {CATALOG}.information_schema.column_tags "
                   f"WHERE schema_name = '{course['SCHEMA']}'").collect():
    print(f"tag: {r['table_name']}.{r['column_name']} {r['tag_name']} = {r['tag_value']}")

In [ ]:
# 6. Read access for the class: see the catalog and the schema, and read every table in it.
schema = f"{CATALOG}.{course['SCHEMA']}"
for statement in [
    f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{CLASS_GROUP}`",
    f"GRANT USE SCHEMA ON SCHEMA {schema} TO `{CLASS_GROUP}`",
    f"GRANT SELECT ON SCHEMA {schema} TO `{CLASS_GROUP}`",
]:
    spark.sql(statement)
print("Granted USE CATALOG, USE SCHEMA and SELECT on", schema, "to", CLASS_GROUP)

In [ ]:
# 7. A choice: run this cell only if each learner should press AI generate in exercise 3.
#    AI generate needs MODIFY on the table, and MODIFY also lets a learner save a comment or
#    change rows. Run cells 2 to 5 again after class, or after anyone saves, to put the tables back.
#    Without this cell, learners use the prepared drafts in exercise 3's drafts.md.
schema = f"{CATALOG}.{course['SCHEMA']}"
for table in ["returns", "third_party_reports", "exam_outcomes"]:
    spark.sql(f"GRANT MODIFY ON TABLE {schema}.{table} TO `{CLASS_GROUP}`")
print("Granted MODIFY on returns, third_party_reports and exam_outcomes to", CLASS_GROUP)